# Демонстрация PixelArt из Python

Нужны Windows, Python/Jupyter и .NET 10 SDK. Ячейки запускают реальное приложение без графического окна, проверяют коды возврата и показывают экспортированный PNG. Дополнительные Python-пакеты, кроме самого Jupyter/IPython, не нужны.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from IPython.display import Image, display

root = Path.cwd().resolve()
while not (root / "PixelArt.sln").exists():
    if root == root.parent:
        raise FileNotFoundError("Откройте notebook внутри папки проекта")
    root = root.parent

local = Path.home() / ".dotnet" / "dotnet.exe"
dotnet = str(local) if local.exists() else shutil.which("dotnet")
if not dotnet:
    raise RuntimeError("Установите .NET 10 SDK")
env = os.environ.copy()
if local.exists():
    env["DOTNET_ROOT"] = str(local.parent)

subprocess.run([dotnet, "build", str(root / "src/PixelArt.App"), "-c", "Release"],
               check=True, env=env)
app = root / "src/PixelArt.App/bin/Release/net10.0-windows/PixelArt.App.dll"
out = root / "artifacts/demo"
out.mkdir(parents=True, exist_ok=True)

def run(*args, check=True):
    result = subprocess.run([dotnet, str(app), *map(str, args)],
                            capture_output=True, text=True, encoding="utf-8", env=env)
    print(result.stdout or result.stderr)
    if check:
        result.check_returncode()
    return result


## 1. Создание проекта через параметры

Файл `.pixelart` — обычный JSON. Здесь создан прозрачный холст размером 24×16.


In [ ]:
import json

run("--new", 24, 16, "--save", out / "blank.pixelart")
data = json.loads((out / "blank.pixelart").read_text(encoding="utf-8-sig"))
assert data["Width"] == 24 and data["Height"] == 16
assert len(data["Pixels"]) == 24 * 16
print("Размер:", data["Width"], "x", data["Height"])


## 2. Экспорт готового рисунка

Каждый исходный пиксель станет квадратом 8×8 без сглаживания; сетка интерфейса в PNG не добавляется.


In [ ]:
sample = root / "samples/landscape.pixelart"
run(sample, "--export", out / "landscape.png", "--scale", 8)
display(Image(filename=str(out / "landscape.png")))


## 3. Поворот и сдвиг

Порядок операций фиксирован: загрузка → поворот → сдвиг → сохранение и экспорт. Освободившаяся область прозрачная, вышедшие за край пиксели обрезаются.


In [ ]:
run(sample, "--rotate", "right", "--move", 2, -3,
    "--save", out / "changed.pixelart",
    "--export", out / "changed.png", "--scale", 8)
display(Image(filename=str(out / "changed.png")))


## 4. Проверка обработки ошибки

Некорректный размер дает код возврата 1, сообщение в stderr и не открывает диалоговых окон.


In [ ]:
result = run("--new", 0, 32, "--save", out / "invalid.pixelart", check=False)
assert result.returncode == 1
assert not (out / "invalid.pixelart").exists()
print("Ошибка обработана корректно")


## 5. Справка

Для запуска графического редактора используйте `run.ps1` без выходных параметров или откройте автономный `PixelArt.App.exe`.


In [ ]:
run("--help");
